# Bank Marketing Campaigns — Data Cleaning

Direct marketing data from a Portuguese bank. Each row is one phone call to one
client, and the target column records whether that client subscribed to a term
deposit afterward.

This notebook is the cleaning pass only. The goal is a dataset that can be read
without the data dictionary open in another tab: readable column names, readable
category values, and one sentinel value handled properly. Exploratory analysis
happens in a separate notebook.

- **Source:** https://www.kaggle.com/datasets/ruthgn/bank-marketing-data-set?resource=download
- **Input:** `bank-direct-marketing-campaigns.csv` — 41,188 rows x 20 columns
- **Output:** `cleaned_bank_marketing.csv` — 41,188 rows x 16 columns

## Data dictionary

Descriptions below are adapted from the original UCI dataset documentation.

#### Client attributes



`age` : Client age in years

`job`: Job type: `admin.`, `blue-collar`, `entrepreneur`, `housemaid`, `management`, `retired`, `self-employed`, `services`, `student`, `technician`, `unemployed`, `unknown`

`marital`:`divorced` (includes widowed), `married`, `single`, `unknown` 

`education`: `basic.4y`, `basic.6y`, `basic.9y`, `high.school`, `illiterate`, `professional.course`, `university.degree`, `unknown`

`default`: Has credit in default: `yes`, `no`, `unknown` 

`housing`: Has a housing loan: `yes`, `no`, `unknown` 

`loan` : Has a personal loan: `yes`, `no`, `unknown` 

#### Last contact of the current campaign

(`contact`)Contact method: `cellular`, `telephone` 

(`month`) Month of last contact, `mar` through `dec`

(`day_of_week`) Weekday of last contact, `mon` through `fri`

(`campaign`) Contacts made to this client during this campaign, including the last one 

#### Previous campaign history

(`pdays`) Days since the client was last contacted in a previous campaign; `999` means never contacted 

(`previous`) Contacts made to this client before this campaign 

(`poutcome` )Outcome of the previous campaign: `failure`, `nonexistent`, `success`

#### Target

(`y`)Did the client subscribe to a term deposit: `yes`, `no` |

#### Macroeconomic indicators — excluded from this notebook

`emp.var.rate`, `cons.price.idx`, `cons.conf.idx`, `euribor3m`, `nr.employed`.
These describe the state of the economy at the time of the call rather than the
client. See the scoping section below for why they come out.

In [76]:
import numpy as np
import pandas as pd

data = pd.read_csv('bank-direct-marketing-campaigns.csv')
data

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41183,73,retired,married,professional.course,no,yes,no,cellular,nov,fri,1,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,yes
41184,46,blue-collar,married,professional.course,no,no,no,cellular,nov,fri,1,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,no
41185,56,retired,married,university.degree,no,yes,no,cellular,nov,fri,2,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,no
41186,44,technician,married,professional.course,no,no,no,cellular,nov,fri,1,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,yes


### Understanding the `education` column

`education` was the first column I had to stop and research rather than clean on
sight. The values `basic.4y`, `basic.6y`, and `basic.9y` don't map onto anything
in an American school system, so my first instinct — reading them as grade levels
or as degree names — was wrong.

They refer to the cycles of Portugal's *ensino básico*: 4, 6, or 9 completed years
of compulsory schooling. So the number is years of education finished, not a
qualification earned. That changes how the column should be ordered and how it
should be read, which is why it's worth pinning down before touching anything else.

In [ ]:
unique_education = data['education'].unique()
unique_education

In [34]:
 data.dtypes

age                 int64
job                   str
marital               str
education             str
default               str
housing               str
loan                  str
contact               str
month                 str
day_of_week           str
campaign            int64
pdays               int64
previous            int64
poutcome              str
emp.var.rate      float64
cons.price.idx    float64
cons.conf.idx     float64
euribor3m         float64
nr.employed       float64
y                     str
dtype: object

## Relabeling Education Column

After watching the education I wanted the education column to be easy to understand, I made the changes to provide more context when reading the education column for better understanding.

### Fields
`basic.4y` -> `basic education - 4 years`

`basic.6y` -> `basic education - 6 years`

`basic.9y` -> `basic education - 9 years`

`high.school` -> `high school / secondary education`

`professional.course` -> `professional / vocational course`

`university.degree` -> `university degree`

`illiterate` and `unknown` were already clear and stayed as they were.




In [35]:
data['education'] = data['education'].str.replace('basic.4y','basic education - 4 years',regex = False)

data['education'] = data['education'].replace('basic.6y','basic education - 6 years', regex = False)

data['education'] = data['education'].replace('basic.9y', 'basic education - 9 years', regex = False)

data['education'] = data['education'].replace('high.school', 'high school / secondary education', regex = False)

data['education'] = data['education'].replace('professional.course','professional / vocational course', regex = False)

data['education'] = data['education'].replace('university.degree','university degree', regex = False)

In [36]:
data['education'].unique()

<StringArray>
[        'basic education - 4 years', 'high school / secondary education',
         'basic education - 6 years',         'basic education - 9 years',
  'professional / vocational course',                           'unknown',
                 'university degree',                        'illiterate']
Length: 8, dtype: str

## Scoping: client-level columns only

The raw file has 20 columns. Five of them — `emp.var.rate`, `cons.price.idx`,
`cons.conf.idx`, `euribor3m`, and `nr.employed` — are macroeconomic indicators:
employment variation rate, consumer price index, consumer confidence index, the
3-month Euribor rate, and the number of people employed.

These are a different kind of variable from the rest. They describe economic
conditions during the period the call was made. Since I don't intend to answer these question based on the state of the economy at the time of the call, I chose to remove them to simplify my analysis.

In [37]:
final_data = data[['age','job','marital','education','default','housing','loan'
                   ,'contact','month','day_of_week','campaign','pdays','previous','poutcome','y']].copy()
final_data

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,campaign,pdays,previous,poutcome,y
0,56,housemaid,married,basic education - 4 years,no,no,no,telephone,may,mon,1,999,0,nonexistent,no
1,57,services,married,high school / secondary education,unknown,no,no,telephone,may,mon,1,999,0,nonexistent,no
2,37,services,married,high school / secondary education,no,yes,no,telephone,may,mon,1,999,0,nonexistent,no
3,40,admin.,married,basic education - 6 years,no,no,no,telephone,may,mon,1,999,0,nonexistent,no
4,56,services,married,high school / secondary education,no,no,yes,telephone,may,mon,1,999,0,nonexistent,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41183,73,retired,married,professional / vocational course,no,yes,no,cellular,nov,fri,1,999,0,nonexistent,yes
41184,46,blue-collar,married,professional / vocational course,no,no,no,cellular,nov,fri,1,999,0,nonexistent,no
41185,56,retired,married,university degree,no,yes,no,cellular,nov,fri,2,999,0,nonexistent,no
41186,44,technician,married,professional / vocational course,no,no,no,cellular,nov,fri,1,999,0,nonexistent,yes


## Renaming columns

Most of the original names are fine and stay as they are: `age`, `job`, `month`,
and `education` say what they hold. Seven don't, and they're the ones that force a
reader back to the data dictionary:

`campaign`, `pdays`,`previous`,`poutcome`,`y`,`housing` ,`loan`. Most of these columns carry no meaning at first glance which is why changing them was very critical.


In [38]:
final_data = final_data.rename(columns = {'campaign':'current_campaign_contacts',
                                          'y':'subscribed',
                                          'pdays':'days_since_previous_contact',
                                          'previous':'previous_campaign_contacts',
                                          'poutcome':'previous_campaign_outcome',
                                          'housing':'housing_loan',
                                          'loan':'personal_loan',
                                          'contact':'contact_type',
                                          'marital':'marital_status',
                                          'default':'credit_default'
                                         })
final_data

,age,job,marital_status,education,credit_default,housing_loan,personal_loan,contact_type,month,day_of_week,current_campaign_contacts,days_since_previous_contact,previous_campaign_contacts,previous_campaign_outcome,subscribed
0,56,housemaid,married,basic education - 4 years,no,no,no,telephone,may,mon,1,999,0,nonexistent,no
1,57,services,married,high school / secondary education,unknown,no,no,telephone,may,mon,1,999,0,nonexistent,no
2,37,services,married,high school / secondary education,no,yes,no,telephone,may,mon,1,999,0,nonexistent,no
3,40,admin.,married,basic education - 6 years,no,no,no,telephone,may,mon,1,999,0,nonexistent,no
4,56,services,married,high school / secondary education,no,no,yes,telephone,may,mon,1,999,0,nonexistent,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41183,73,retired,married,professional / vocational course,no,yes,no,cellular,nov,fri,1,999,0,nonexistent,yes
41184,46,blue-collar,married,professional / vocational course,no,no,no,cellular,nov,fri,1,999,0,nonexistent,no
41185,56,retired,married,university degree,no,yes,no,cellular,nov,fri,2,999,0,nonexistent,no
41186,44,technician,married,professional / vocational course,no,no,no,cellular,nov,fri,1,999,0,nonexistent,yes


In [39]:
final_data['month'].unique()

<StringArray>
['may', 'jun', 'jul', 'aug', 'oct', 'nov', 'dec', 'mar', 'apr', 'sep']
Length: 10, dtype: str

###  Month Column

The month values were abbreviated (e.g., may, jun, jul). I expanded the abbreviations to full month names to make the column easier to interpret. I then converted the column to an ordered categorical variable so the months could be displayed chronologically during analysis.

In [40]:
final_data['month'] = final_data['month'].replace(['may',
                                                    'jun', 
                                                    'jul', 
                                                    'aug',
                                                    'oct', 
                                                    'nov', 
                                                    'dec', 
                                                    'mar', 
                                                    'apr', 
                                                    'sep'], ['may',
                                                             'june', 
                                                             'july', 
                                                             'august', 
                                                             'october',
                                                             'november',
                                                             'december','march','april','september'], regex=False)

In [41]:
final_data['month'].unique()

<StringArray>
[      'may',      'june',      'july',    'august',   'october',  'november',
  'december',     'march',     'april', 'september']
Length: 10, dtype: str

In [42]:
month_order = ['march', 
               'april',
               'may','june','july','august','september','october', 'november','december']
final_data['month'] = pd.Categorical( final_data['month'], categories = month_order, ordered=True)

In [43]:
final_data['day_of_week'].unique()

<StringArray>
['mon', 'tue', 'wed', 'thu', 'fri']
Length: 5, dtype: str

###  Day_of_Week Column

The Day_of_Week column was carrying values that were abbreviated and needed
better context in my opinion, as I stated previously understanding the 
dataset before starting any type of work in my first goal.

In [44]:
final_data['day_of_week'] = final_data['day_of_week'].replace(['mon', 'tue', 'wed', 'thu', 'fri'],
                                                             ['monday','tuesday','wednesday','thursday','friday'], regex=False)

In [45]:
days_of_week_order = ['monday','tuesday','wednesday','thursday','friday']
final_data['day_of_week'] = pd.Categorical(final_data['day_of_week'],categories=days_of_week_order,ordered=True)

In [46]:
sorted_month_and_day_of_week = final_data.sort_values(by=['month','day_of_week'],ascending=True)
sorted_month_and_day_of_week

,age,job,marital_status,education,credit_default,housing_loan,personal_loan,contact_type,month,day_of_week,current_campaign_contacts,days_since_previous_contact,previous_campaign_contacts,previous_campaign_outcome,subscribed
27690,26,student,single,basic education - 9 years,no,yes,no,cellular,march,monday,4,999,0,nonexistent,yes
27691,38,admin.,single,university degree,no,yes,no,cellular,march,monday,2,999,0,nonexistent,yes
27692,26,student,single,basic education - 9 years,no,no,no,cellular,march,monday,4,999,0,nonexistent,no
27693,26,student,single,basic education - 9 years,no,yes,no,telephone,march,monday,4,999,0,nonexistent,no
27694,30,management,married,university degree,no,yes,no,telephone,march,monday,7,999,0,nonexistent,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39067,23,blue-collar,single,high school / secondary education,no,yes,no,cellular,december,friday,6,999,0,nonexistent,no
39068,30,self-employed,single,university degree,no,yes,yes,cellular,december,friday,2,6,1,success,no
39069,55,entrepreneur,divorced,basic education - 9 years,no,no,no,cellular,december,friday,7,999,0,nonexistent,yes
39070,36,admin.,married,university degree,no,yes,no,telephone,december,friday,4,999,0,nonexistent,no


In [47]:
final_data['days_since_previous_contact'].unique()


array([999,   6,   4,   3,   5,   1,   0,  10,   7,   8,   9,  11,   2,
        12,  13,  14,  15,  16,  21,  17,  18,  22,  25,  26,  19,  27,
        20])

### Previous Contact Information

The `days_since_previous_contact` column used `999` to indicate that a client had not been contacted in a previous campaign. Since `999` does not represent an actual number of days, I replaced it with `NaN` and created a separate `previously_contacted` column to explicitly identify whether a previous contact existed.

In [48]:
final_data["days_since_previous_contact"] = final_data["days_since_previous_contact"].replace(999, np.nan)


final_data["previously_contacted"] = final_data["days_since_previous_contact"].notna()


In [49]:
final_data['job'].unique()

<StringArray>
[    'housemaid',      'services',        'admin.',   'blue-collar',
    'technician',       'retired',    'management',    'unemployed',
 'self-employed',       'unknown',  'entrepreneur',       'student']
Length: 12, dtype: str

## `job` and `unknown` values

`job` spells one of the category fields `admin.` with a trailling period. It makes the field more prone error later in case you want to use and forget it has a period trailling, In that case I've chosen to remove it for easier analysis.

`unknown` which appears in `job`, `marital_status`, `education`,
`credit_default`, `housing_loan`, and `personal_loan` — with `credit_default`
carrying a large share of them.

In [50]:
final_data['job'] = final_data['job'].str.replace('admin.','admin', regex=False)

In [51]:
final_data['job'].unique()

<StringArray>
[    'housemaid',      'services',         'admin',   'blue-collar',
    'technician',       'retired',    'management',    'unemployed',
 'self-employed',       'unknown',  'entrepreneur',       'student']
Length: 12, dtype: str

In [52]:
final_data['marital_status'].unique()

<StringArray>
['married', 'single', 'divorced', 'unknown']
Length: 4, dtype: str

## Null check

one columns reports nulls: `days_since_previous_contact` which is the column that was carrying 999 for never contacted. Further analysis is still needed to confirm the `39,673` are all the ones that we changed were the people that were not `previously contacted`.

In [53]:
null_data = final_data.isnull().sum()
null_data

age                                0
job                                0
marital_status                     0
education                          0
credit_default                     0
housing_loan                       0
personal_loan                      0
contact_type                       0
month                              0
day_of_week                        0
current_campaign_contacts          0
days_since_previous_contact    39673
previous_campaign_contacts         0
previous_campaign_outcome          0
subscribed                         0
previously_contacted               0
dtype: int64

In [66]:
final_data.query('previously_contacted == False').count()

age                            39673
job                            39673
marital_status                 39673
education                      39673
credit_default                 39673
housing_loan                   39673
personal_loan                  39673
contact_type                   39673
month                          39673
day_of_week                    39673
current_campaign_contacts      39673
days_since_previous_contact        0
previous_campaign_contacts     39673
previous_campaign_outcome      39673
subscribed                     39673
previously_contacted           39673
dtype: int64

In [28]:
final_data

,age,job,marital_status,education,credit_default,housing_loan,personal_loan,contact_type,month,day_of_week,current_campaign_contacts,days_since_previous_contact,previous_campaign_contacts,previous_campaign_outcome,subscribed,previously_contacted
0,56,housemaid,married,basic education - 4 years,no,no,no,telephone,may,monday,1,NaN,0,nonexistent,no,False
1,57,services,married,high school / secondary education,unknown,no,no,telephone,may,monday,1,NaN,0,nonexistent,no,False
2,37,services,married,high school / secondary education,no,yes,no,telephone,may,monday,1,NaN,0,nonexistent,no,False
3,40,admin,married,basic education - 6 years,no,no,no,telephone,may,monday,1,NaN,0,nonexistent,no,False
4,56,services,married,high school / secondary education,no,no,yes,telephone,may,monday,1,NaN,0,nonexistent,no,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41183,73,retired,married,professional / vocational course,no,yes,no,cellular,november,friday,1,NaN,0,nonexistent,yes,False
41184,46,blue-collar,married,professional / vocational course,no,no,no,cellular,november,friday,1,NaN,0,nonexistent,no,False
41185,56,retired,married,university degree,no,yes,no,cellular,november,friday,2,NaN,0,nonexistent,no,False
41186,44,technician,married,professional / vocational course,no,no,no,cellular,november,friday,1,NaN,0,nonexistent,yes,False


In [31]:
final_data.to_csv("cleaned_bank_marketing.csv", index=False)